# Пример использования `fedstat`

Библиотека скачивает данные показателей с fedstat.ru (ЕМИСС) и отдаёт нормализованный `pandas.DataFrame`.

> Нужен доступ к fedstat.ru (российский IP). Показатель в примере — **31452** «Средняя цена 1 кв. м жилья» (id берётся из URL вида `https://www.fedstat.ru/indicator/31452`).

## 1. Какие фильтры есть у показателя

In [1]:
import fedstat

filters_table = fedstat.list_filters("31452")
filters_table.head(20)

,filter_field_title,filter_value_title,filter_field_id,filter_value_id,filter_field_object_ids
0,Показатель,Средняя цена 1 кв. м общей площади квартир на...,0,31452,filterObjectIds
1,Год,2000,3,2000,columnObjectIds
2,Год,2001,3,2001,columnObjectIds
3,Год,2002,3,2002,columnObjectIds
4,Год,2003,3,2003,columnObjectIds
5,Год,2004,3,2004,columnObjectIds
6,Год,2005,3,2005,columnObjectIds
7,Год,2006,3,2006,columnObjectIds
8,Год,2007,3,2007,columnObjectIds
9,Год,2008,3,2008,columnObjectIds


## 2. Шаблон фильтров
`filter_template` возвращает словарь со всеми полями показателя (значение `"*"` = все значения). Правим только то, что нужно сузить; пропущенные поля берутся целиком.

In [2]:
f = fedstat.filter_template("31452")
f  # посмотреть все доступные ключи

{'Показатель': '*',
 'Год': '*',
 'Классификатор объектов административно-территориального деления (ОКАТО)': '*',
 'Единица измерения': '*',
 'Период': '*',
 'Рынок жилья': '*',
 'Типы квартир': '*'}

In [3]:
f["Год"] = "2023"
f["Рынок жилья"] = "Первичный рынок жилья"
f["Типы квартир"] = "Все типы квартир"
f["Классификатор объектов административно-территориального деления (ОКАТО)"] = "Российская Федерация"
f

{'Показатель': '*',
 'Год': '2023',
 'Классификатор объектов административно-территориального деления (ОКАТО)': 'Российская Федерация',
 'Единица измерения': '*',
 'Период': '*',
 'Рынок жилья': 'Первичный рынок жилья',
 'Типы квартир': 'Все типы квартир'}

## 3. Скачать данные — нормализованный («длинный») DataFrame

In [ ]:
df = fedstat.load("31452", filters=f)
print(df.shape)
df.head(10)

DownloadError: Не удалось скачать данные индикатора 31452 за 3 попыток. Последняя ошибка: fedstat вернул 503 (Service Unavailable): сервер перегружен. Обычно 503/302 — временная нестабильность fedstat; повторите позже или увеличьте retry_max_times/retry_pause. Если ошибка повторяется стабильно — проверьте значения фильтров.

In [5]:
df = fedstat.load("31452", filters=f, retry_max_times=8, retry_pause=5)

KeyboardInterrupt: 

## 4. «Широкий» вид: кварталы по столбцам

In [5]:
wide = fedstat.to_wide(df, columns="PERIOD", values="VALUE", index="TIME")
wide

NameError: name 'df' is not defined

## 5. Сохранить в CSV / Excel

In [ ]:
df.to_csv("cena.csv", index=False)
df.to_excel("cena.xlsx", index=False)
print("сохранено: cena.csv, cena.xlsx")

---
### Подсказки
- Неверное имя поля/значения -> понятная ошибка `FilterError` с подсказкой похожего варианта.
- `load(..., with_codes=True)` — добавить исходные коды измерений.
- `load(..., drop_empty=True)` — выбросить строки без значения.
- Низкоуровневые шаги доступны отдельно: `fedstat.get_data_ids("31452")`.